# 11: Batch Correction - Clear Comparison

**3 Simple Questions:**
1. Can we predict Control vs Chemo at t=0? (should be ~50% = no biological signal)
2. Does ComBat help at t=0?
3. Does ComBat help at t=14? (where biology IS present)

**Setup:**
- Use ALL cells (no arbitrary sampling)
- Train on: flattened images → PCA 50D → SVM
- Compare: raw vs ComBat-corrected
- Show t-SNE visualization of batch effect

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('/baldig/bioprojects2/emartinl/chemores/preprocessed_phasor')
RESULTS_DIR = Path('./results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FILE_INFO = [
    ('CNTL-MB231', 'Control'),
    ('TAMO-MB231', 'Chemoresistant'),
    ('CNTL_75uM_p1', 'Control'),
    ('CNTL_75uM_p2', 'Control'),
    ('CNTL_75uM_p3', 'Control'),
    ('CNTL_75uM_p4', 'Control'),
    ('TAMO_p1', 'Chemoresistant'),
    ('TAMO_p2', 'Chemoresistant'),
]

## Helpers

In [3]:
def load_all_cells(timepoint):
    """Load ALL cells at timepoint (no sampling)."""
    all_meta = []
    file_handles = {}
    
    for name, group in FILE_INFO:
        meta = pd.read_csv(DATA_DIR / f'{name}_cells_meta.csv')
        mask = meta['time'] == timepoint
        idx = np.where(mask)[0]
        
        if len(idx) == 0:
            continue
        
        X_mmap = np.load(DATA_DIR / f'{name}_cells.npy', mmap_mode='r')
        
        meta_t = meta.loc[mask].copy()
        meta_t['file'] = name
        meta_t['group'] = group
        meta_t['cell_idx'] = idx
        meta_t = meta_t.reset_index(drop=True)
        
        all_meta.append(meta_t)
        file_handles[name] = X_mmap
    
    meta_all = pd.concat(all_meta, ignore_index=True)
    meta_all['batch_id'] = pd.factorize(meta_all['file'])[0]
    
    return meta_all, file_handles


def load_images(meta, file_handles):
    """Load cell images from memory-mapped arrays."""
    X_raw = []
    for idx, row in meta.iterrows():
        X_mmap = file_handles[row['file']]
        cell_idx = row['cell_idx']
        img = np.array(X_mmap[cell_idx])
        X_raw.append(img)
    return np.array(X_raw)


def combat_correction(X, batch_ids):
    """Simple empirical Bayes batch correction."""
    X = np.array(X, dtype=float)
    X_corrected = X.copy()
    
    for batch in np.unique(batch_ids):
        mask = batch_ids == batch
        batch_data = X[mask]
        
        grand_mean = X.mean(axis=0)
        grand_var = X.var(axis=0) + 1e-8
        batch_mean = batch_data.mean(axis=0)
        batch_var = batch_data.var(axis=0) + 1e-8
        
        X_corrected[mask] = (batch_data - batch_mean) / np.sqrt(batch_var) * np.sqrt(grand_var) + grand_mean
    
    return X_corrected

## QUESTION 1: Can we predict at t=0? (baseline = should be ~50%)

In [4]:
print('Loading t=0 cells...')
meta_t0, fh_t0 = load_all_cells(0)
print(f'  Loaded {len(meta_t0)} cells')

X_raw_t0 = load_images(meta_t0, fh_t0)
print(f'  Shape: {X_raw_t0.shape}')

# Flatten and standardize
X_flat_t0 = X_raw_t0.reshape(X_raw_t0.shape[0], -1)
scaler_t0 = StandardScaler()
X_scaled_t0 = scaler_t0.fit_transform(X_flat_t0)
print(f'  Flattened: {X_scaled_t0.shape}')

# PCA to 50D
pca_t0 = PCA(n_components=50)
X_pca_t0 = pca_t0.fit_transform(X_scaled_t0)
print(f'  After PCA: {X_pca_t0.shape}')

batch_ids_t0 = meta_t0['batch_id'].values
y_t0 = (meta_t0['group'] == 'Chemoresistant').astype(int).values

print(f'\n  Control: {(y_t0==0).sum()}, Chemo: {(y_t0==1).sum()}')

Loading t=0 cells...
  Loaded 2365 cells
  Shape: (2365, 7, 512, 512)
  Flattened: (2365, 1835008)
  After PCA: (2365, 50)

  Control: 865, Chemo: 1500


In [5]:
# Train on RAW t=0
sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)

results_t0_raw = []
for fold, (train_idx, test_idx) in enumerate(sgkf.split(X_pca_t0, y_t0, batch_ids_t0), 1):
    clf = SVC(kernel='rbf', random_state=42)
    clf.fit(X_pca_t0[train_idx], y_t0[train_idx])
    acc = clf.score(X_pca_t0[test_idx], y_t0[test_idx])
    results_t0_raw.append(acc)

print('t=0 RAW (no correction):')
print(f'  Fold accuracies: {[f"{x:.3f}" for x in results_t0_raw]}')
print(f'  Mean: {np.mean(results_t0_raw):.3f} ± {np.std(results_t0_raw):.3f}')
print(f'  → Should be ~50% (no biology at t=0)')

t=0 RAW (no correction):
  Fold accuracies: ['0.187', '0.150', '0.368']
  Mean: 0.235 ± 0.095
  → Should be ~50% (no biology at t=0)


## QUESTION 2: Does ComBat help at t=0?

In [6]:
print('Applying ComBat to t=0...')
X_corrected_pca_t0 = combat_correction(X_pca_t0, batch_ids_t0)
print('  Done')

# Train on CORRECTED t=0
results_t0_corr = []
for fold, (train_idx, test_idx) in enumerate(sgkf.split(X_pca_t0, y_t0, batch_ids_t0), 1):
    clf = SVC(kernel='rbf', random_state=42)
    clf.fit(X_corrected_pca_t0[train_idx], y_t0[train_idx])
    acc = clf.score(X_corrected_pca_t0[test_idx], y_t0[test_idx])
    results_t0_corr.append(acc)

print('\nt=0 CORRECTED (ComBat):')
print(f'  Fold accuracies: {[f"{x:.3f}" for x in results_t0_corr]}')
print(f'  Mean: {np.mean(results_t0_corr):.3f} ± {np.std(results_t0_corr):.3f}')

print(f'\nComparison at t=0:')
print(f'  Raw:       {np.mean(results_t0_raw):.3f}')
print(f'  Corrected: {np.mean(results_t0_corr):.3f}')
print(f'  Δ:         {(np.mean(results_t0_corr) - np.mean(results_t0_raw)):+.3f}')
if np.mean(results_t0_corr) > np.mean(results_t0_raw):
    print('  → ComBat HELPS')
else:
    print('  → ComBat HURTS (or has no effect)')

Applying ComBat to t=0...
  Done

t=0 CORRECTED (ComBat):
  Fold accuracies: ['0.580', '0.169', '0.434']
  Mean: 0.394 ± 0.170

Comparison at t=0:
  Raw:       0.235
  Corrected: 0.394
  Δ:         +0.159
  → ComBat HELPS


## QUESTION 3: Does ComBat help at t=14? (where biology exists)

In [7]:
print('Loading t=14 cells...')
meta_t14, fh_t14 = load_all_cells(14)
if len(meta_t14) == 0:
    print('  No t=14 cells found')
else:
    print(f'  Loaded {len(meta_t14)} cells')
    
    X_raw_t14 = load_images(meta_t14, fh_t14)
    print(f'  Shape: {X_raw_t14.shape}')
    
    # Same pipeline as t=0
    X_flat_t14 = X_raw_t14.reshape(X_raw_t14.shape[0], -1)
    scaler_t14 = StandardScaler()
    X_scaled_t14 = scaler_t14.fit_transform(X_flat_t14)
    
    pca_t14 = PCA(n_components=50)
    X_pca_t14 = pca_t14.fit_transform(X_scaled_t14)
    print(f'  After PCA: {X_pca_t14.shape}')
    
    batch_ids_t14 = meta_t14['batch_id'].values
    y_t14 = (meta_t14['group'] == 'Chemoresistant').astype(int).values
    
    print(f'  Control: {(y_t14==0).sum()}, Chemo: {(y_t14==1).sum()}')

Loading t=14 cells...
  Loaded 2288 cells
  Shape: (2288, 7, 512, 512)
  After PCA: (2288, 50)
  Control: 913, Chemo: 1375


In [8]:
# Train on RAW t=14
results_t14_raw = []
for fold, (train_idx, test_idx) in enumerate(sgkf.split(X_pca_t14, y_t14, batch_ids_t14), 1):
    clf = SVC(kernel='rbf', random_state=42)
    clf.fit(X_pca_t14[train_idx], y_t14[train_idx])
    acc = clf.score(X_pca_t14[test_idx], y_t14[test_idx])
    results_t14_raw.append(acc)

print('t=14 RAW (no correction):')
print(f'  Fold accuracies: {[f"{x:.3f}" for x in results_t14_raw]}')
print(f'  Mean: {np.mean(results_t14_raw):.3f} ± {np.std(results_t14_raw):.3f}')

t=14 RAW (no correction):
  Fold accuracies: ['0.374', '0.062', '0.228']
  Mean: 0.221 ± 0.127


In [9]:
# Apply ComBat at t=14
X_corrected_pca_t14 = combat_correction(X_pca_t14, batch_ids_t14)

# Train on CORRECTED t=14
results_t14_corr = []
for fold, (train_idx, test_idx) in enumerate(sgkf.split(X_pca_t14, y_t14, batch_ids_t14), 1):
    clf = SVC(kernel='rbf', random_state=42)
    clf.fit(X_corrected_pca_t14[train_idx], y_t14[train_idx])
    acc = clf.score(X_corrected_pca_t14[test_idx], y_t14[test_idx])
    results_t14_corr.append(acc)

print('t=14 CORRECTED (ComBat):')
print(f'  Fold accuracies: {[f"{x:.3f}" for x in results_t14_corr]}')
print(f'  Mean: {np.mean(results_t14_corr):.3f} ± {np.std(results_t14_corr):.3f}')

print(f'\nComparison at t=14:')
print(f'  Raw:       {np.mean(results_t14_raw):.3f}')
print(f'  Corrected: {np.mean(results_t14_corr):.3f}')
print(f'  Δ:         {(np.mean(results_t14_corr) - np.mean(results_t14_raw)):+.3f}')
if np.mean(results_t14_corr) > np.mean(results_t14_raw) + 0.05:
    print('  → ComBat HELPS significantly')
elif np.mean(results_t14_corr) > np.mean(results_t14_raw):
    print('  → ComBat slightly helps')
else:
    print('  → ComBat does not help')

t=14 CORRECTED (ComBat):
  Fold accuracies: ['0.575', '0.062', '0.177']
  Mean: 0.271 ± 0.220

Comparison at t=14:
  Raw:       0.221
  Corrected: 0.271
  Δ:         +0.050
  → ComBat HELPS significantly
